# Pandas III — Análisis avanzado con dataset real

## Objetivos
- Preparar un dataset real para análisis.
- Hacer checks contables y temporales.
- Dominar `groupby`, `agg`, `transform`, series temporales y `pivot_table`.
- Llegar a reporting útil sin intentar cubrir demasiado.

## Nota de sesión
Este notebook está recortado para que dé tiempo a cerrar lo pendiente de Pandas II y llegar a lo esencial de Pandas III.

---
## 0. Setup

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', '{:,.2f}'.format)


---
## 1. Carga e inspección inicial

In [2]:
path = 'supermarket_sales - Sheet1.csv'
df = pd.read_csv(path)

print('shape:', df.shape)
display(df.head())


shape: (1000, 17)


,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.14,548.97,1/5/2019,13:08,Ewallet,522.83,4.76,26.14,9.10
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.82,80.22,3/8/2019,10:29,Cash,76.40,4.76,3.82,9.60
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.22,340.53,3/3/2019,13:23,Credit card,324.31,4.76,16.22,7.40
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.29,489.05,1/27/2019,20:33,Ewallet,465.76,4.76,23.29,8.40
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.21,634.38,2/8/2019,10:37,Ewallet,604.17,4.76,30.21,5.30


### Demo guiada 1: inspección rápida

Esta parte la puedes enseñar tú sin convertirla en ejercicio largo.

In [3]:
df.info()

display(df[['Invoice ID', 'City', 'Product line', 'Payment']].nunique())
print('Exact duplicated rows:', df.duplicated().sum())


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               1000 non-null   str    
 1   Branch                   1000 non-null   str    
 2   City                     1000 non-null   str    
 3   Customer type            1000 non-null   str    
 4   Gender                   1000 non-null   str    
 5   Product line             1000 non-null   str    
 6   Unit price               1000 non-null   float64
 7   Quantity                 1000 non-null   int64  
 8   Tax 5%                   1000 non-null   float64
 9   Total                    1000 non-null   float64
 10  Date                     1000 non-null   str    
 11  Time                     1000 non-null   str    
 12  Payment                  1000 non-null   str    
 13  cogs                     1000 non-null   float64
 14  gross margin percentage  1000 non-nu

Invoice ID      1000
City               3
Product line       6
Payment            3
dtype: int64

Exact duplicated rows: 0


---
## 2. Tipos, datetime y checks básicos

Aquí dejamos listo el dataset y hacemos las validaciones mínimas, pero en modo guiado para no comernos la sesión.

### Ejemplo: parseo de fechas y features temporales

In [4]:
df2 = df.copy()
df2['Date'] = pd.to_datetime(df2['Date'])
df2['datetime'] = pd.to_datetime(df2['Date'].dt.strftime('%Y-%m-%d') + ' ' + df2['Time'])
df2['month'] = df2['datetime'].dt.month
df2['weekday'] = df2['datetime'].dt.day_name()
df2['hour'] = df2['datetime'].dt.hour

display(df2[['Date', 'Time', 'datetime', 'month', 'weekday', 'hour']].head())


,Date,Time,datetime,month,weekday,hour
0,2019-01-05,13:08,2019-01-05 13:08:00,1,Saturday,13
1,2019-03-08,10:29,2019-03-08 10:29:00,3,Friday,10
2,2019-03-03,13:23,2019-03-03 13:23:00,3,Sunday,13
3,2019-01-27,20:33,2019-01-27 20:33:00,1,Sunday,20
4,2019-02-08,10:37,2019-02-08 10:37:00,2,Friday,10


### Demo guiada 2: categóricas y memoria

In [5]:
mem_before = df2.memory_usage(deep=True).sum()
cat_cols = ['Branch', 'City', 'Customer type', 'Gender', 'Product line', 'Payment']
df2[cat_cols] = df2[cat_cols].astype('category')
mem_after = df2.memory_usage(deep=True).sum()

print('Memory before:', mem_before)
print('Memory after :', mem_after)
print('Reduction %   :', round((1 - mem_after / mem_before) * 100, 2))
display(df2.dtypes)


Memory before: 597725
Memory after : 265435
Reduction %   : 55.59


Invoice ID                            str
Branch                           category
City                             category
Customer type                    category
Gender                           category
Product line                     category
Unit price                        float64
Quantity                            int64
Tax 5%                            float64
Total                             float64
Date                       datetime64[us]
Time                                  str
Payment                          category
cogs                              float64
gross margin percentage           float64
gross income                      float64
Rating                            float64
datetime                   datetime64[us]
month                               int32
weekday                               str
hour                                int32
dtype: object

### Demo guiada 3: sanity checks

In [ ]:
tol = 1e-6
fail_total = df2.loc[~np.isclose(df2['Total'], df2['cogs'] + df2['Tax 5%'], atol=tol)]
fail_income = df2.loc[~np.isclose(df2['gross income'], df2['cogs'] * 0.05, atol=tol)]

print('fail_total rows :', fail_total.shape[0])
print('fail_income rows:', fail_income.shape[0])
display(fail_total.head())
display(fail_income.head())


---
## 3. `groupby` + `agg` para KPIs

Este es uno de los bloques clave de la sesión.

### Ejemplo: KPIs por ciudad

In [6]:
city_kpis = df2.groupby('City', as_index=False).agg(
    total_sales=('Total', 'sum'),
    tx_count=('Invoice ID', 'count'),
    avg_ticket=('Total', 'mean'),
    avg_rating=('Rating', 'mean'),
    total_profit=('gross income', 'sum')
).sort_values('total_sales', ascending=False)

display(city_kpis)


,City,total_sales,tx_count,avg_ticket,avg_rating,total_profit
1,Naypyitaw,"110,568.71",328,337.10,7.07,"5,265.18"
2,Yangon,"106,200.37",340,312.35,7.03,"5,057.16"
0,Mandalay,"106,197.67",332,319.87,6.82,"5,057.03"


### Ejercicio 1
Crea tablas KPI con las mismas métricas por:
1. `Branch`
2. `Product line`
3. `Payment`

In [13]:
# KPI por Branch, Product line y Payment

# 1. KPIs por sucursal (Branch)
kpi_branch = (
    df.groupby("Branch")
      .agg(
          total_sales=("Total", "sum"),
          avg_sales=("Total", "mean"),
          total_transactions=("Invoice ID", "count"),
          avg_rating=("Rating", "mean"),
          total_quantity=("Quantity", "sum"),
          gross_income=("gross income", "sum")
      )
      .round(2)
      .sort_values("total_sales", ascending=False)
)

print("=== KPI POR BRANCH ===")
print(kpi_branch)


# 2. KPIs por línea de producto (Product line)
kpi_product_line = (
    df.groupby("Product line")
      .agg(
          total_sales=("Total", "sum"),
          avg_sales=("Total", "mean"),
          total_transactions=("Invoice ID", "count"),
          avg_rating=("Rating", "mean"),
          total_quantity=("Quantity", "sum"),
          gross_income=("gross income", "sum")
      )
      .round(2)
      .sort_values("total_sales", ascending=False)
)

print("\n=== KPI POR PRODUCT LINE ===")
print(kpi_product_line)


# 3. KPIs por método de pago (Payment)
kpi_payment = (
    df.groupby("Payment")
      .agg(
          total_sales=("Total", "sum"),
          avg_sales=("Total", "mean"),
          total_transactions=("Invoice ID", "count"),
          avg_rating=("Rating", "mean"),
          total_quantity=("Quantity", "sum"),
          gross_income=("gross income", "sum")
      )
      .round(2)
      .sort_values("total_sales", ascending=False)
)

print("\n=== KPI POR PAYMENT ===")
print(kpi_payment)

=== KPI POR BRANCH ===
        total_sales  avg_sales  total_transactions  avg_rating  total_quantity  gross_income
Branch                                                                                      
C        110,568.71     337.10                 328        7.07            1831      5,265.18
A        106,200.37     312.35                 340        7.03            1859      5,057.16
B        106,197.67     319.87                 332        6.82            1820      5,057.03

=== KPI POR PRODUCT LINE ===
                        total_sales  avg_sales  total_transactions  avg_rating  total_quantity  gross_income
Product line                                                                                                
Food and beverages        56,144.84     322.67                 174        7.11             952      2,673.56
Sports and travel         55,122.83     332.07                 166        6.92             920      2,624.90
Electronic accessories    54,337.53     319.63

---
## 4. `transform`

`transform` te permite devolver a cada fila una métrica calculada a nivel grupo.

### Ejemplo: share del ticket dentro de su ciudad

In [ ]:
fs = df2.copy()
fs['city_total_sales'] = fs.groupby('City')['Total'].transform('sum')
fs['ticket_share_city'] = fs['Total'] / fs['city_total_sales']

display(fs[['City', 'Total', 'city_total_sales', 'ticket_share_city']].head())


### Ejercicio 2
1. Crea `product_line_total_sales` con `transform`.
2. Crea `ticket_share_product_line`.
3. Ordena por `Product line` y `ticket_share_product_line` descendente.

In [15]:
from IPython.display import display

# 1. Total de ventas por Product line usando transform()
df["product_line_total_sales"] = (
    df.groupby("Product line")["Total"]
      .transform("sum")
)

# 2. Peso de cada ticket sobre el total de su Product line
df["ticket_share_product_line"] = (
    df["Total"] / df["product_line_total_sales"]
)

# 3. Ordenar por Product line y ticket_share_product_line descendente
df_sorted = (
    df.sort_values(
        by=["Product line", "ticket_share_product_line"],
        ascending=[True, False]
    )
)

# 4. Mostrar resultado en formato tabla bonito en Jupyter
display(
    df_sorted[
        [
            "Invoice ID",
            "Product line",
            "Total",
            "product_line_total_sales",
            "ticket_share_product_line"
        ]
    ]
    .head(20)
    .round({
        "Total": 2,
        "product_line_total_sales": 2,
        "ticket_share_product_line": 4
    })
)

,Invoice ID,Product line,Total,product_line_total_sales,ticket_share_product_line
209,817-69-8206,Electronic accessories,942.45,"54,337.53",0.02
105,704-48-3927,Electronic accessories,931.04,"54,337.53",0.02
988,267-62-7380,Electronic accessories,864.57,"54,337.53",0.02
109,861-77-0145,Electronic accessories,860.68,"54,337.53",0.02
120,638-60-7125,Electronic accessories,836.30,"54,337.53",0.02
928,431-66-2305,Electronic accessories,833.96,"54,337.53",0.02
457,533-33-5337,Electronic accessories,833.60,"54,337.53",0.02
779,457-94-0464,Electronic accessories,830.37,"54,337.53",0.02
314,227-07-4446,Electronic accessories,820.36,"54,337.53",0.02
829,416-17-9926,Electronic accessories,779.31,"54,337.53",0.01


---
## 5. Series temporales

Nos quedamos con la parte más útil: agregación temporal y patrones.

### Ejemplo: serie diaria y rolling de 7 días

In [ ]:
ts = df2.set_index('datetime').sort_index()
daily_sales = ts['Total'].resample('D').sum().fillna(0)
daily_roll7 = daily_sales.rolling(7, min_periods=1).mean()

display(pd.DataFrame({'daily_sales': daily_sales, 'roll7': daily_roll7}).head(10))


### Ejercicio 3
1. Construye una serie semanal (`W`) de ventas.
2. Calcula la media móvil de 4 semanas.
3. Devuelve un DataFrame con `weekly_sales` y `rolling_4w`.

In [16]:
from IPython.display import display
import pandas as pd

# Asegurarnos de que Date es datetime
df["Date"] = pd.to_datetime(df["Date"])

# 1. Serie semanal de ventas
weekly_sales = (
    df.set_index("Date")
      .resample("W")["Total"]
      .sum()
)

# 2. Media móvil de 4 semanas
rolling_4w = (
    weekly_sales
    .rolling(window=4)
    .mean()
)

# 3. DataFrame final
weekly_sales_df = pd.DataFrame({
    "weekly_sales": weekly_sales,
    "rolling_4w": rolling_4w
}).round(2)

# Mostrar resultado
display(weekly_sales_df)

,weekly_sales,rolling_4w
Date,,
2019-01-06,"17,543.39",NaN
2019-01-13,"24,461.20",NaN
2019-01-20,"28,693.36",NaN
2019-01-27,"29,286.88","24,996.21"
2019-02-03,"28,360.45","27,700.47"
2019-02-10,"27,101.83","28,360.63"
2019-02-17,"25,563.59","27,578.19"
2019-02-24,"17,328.66","24,588.63"
2019-03-03,"29,219.72","24,803.45"


### Ejercicio 4
1. Calcula ventas totales por `weekday`.
2. Calcula ventas totales por `hour`.
3. Encuentra la combinación `weekday` + `hour` con mayor venta total.

In [17]:
import pandas as pd
from IPython.display import display

# 1. Cargar los datos
df = pd.read_csv('supermarket_sales - Sheet1.csv')

# Convertir Date y Time a objetos datetime
df['Date'] = pd.to_datetime(df['Date'])
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M').dt.hour

# Crear la columna weekday (0=Lunes, 6=Domingo)
# Usamos day_name() para que sea más legible
df['weekday'] = df['Date'].dt.day_name()

# --- Cálculos ---

# 1. Ventas totales por weekday
sales_by_weekday = df.groupby('weekday')['Total'].sum().sort_values(ascending=False)

# 2. Ventas totales por hour
sales_by_hour = df.groupby('Time')['Total'].sum().sort_values(ascending=False)

# 3. Combinación weekday + hour con mayor venta
sales_combined = df.groupby(['weekday', 'Time'])['Total'].sum().reset_index()
best_combo = sales_combined.loc[sales_combined['Total'].idxmax()]

# --- Mostrar resultados ---
print("### 1. Ventas Totales por Día de la Semana")
display(sales_by_weekday.to_frame())

print("\n### 2. Ventas Totales por Hora")
display(sales_by_hour.to_frame())

print("\n### 3. Combinación con Mayor Venta")
display(best_combo.to_frame().T)

### 1. Ventas Totales por Día de la Semana


,Total
weekday,
Saturday,"56,120.81"
Tuesday,"51,482.25"
Thursday,"45,349.25"
Sunday,"44,457.89"
Friday,"43,926.34"
Wednesday,"43,731.14"
Monday,"37,899.08"



### 2. Ventas Totales por Hora


,Total
Time,
19,"39,699.51"
13,"34,723.23"
10,"31,421.48"
15,"31,179.51"
14,"30,828.40"
11,"30,377.33"
12,"26,065.88"
18,"26,030.34"
16,"25,226.32"



### 3. Combinación con Mayor Venta


,weekday,Time,Total
64,Tuesday,19,"9,198.67"


---
## 6. Pivot tables

Este bloque suele dar muchísimo retorno práctico en clase.

### Ejemplo: City × Payment

In [ ]:
pv = pd.pivot_table(df2, values='Total', index='City', columns='Payment', aggfunc='sum').fillna(0)
pv['Total'] = pv.sum(axis=1)
pv = pv.sort_values('Total', ascending=False)

display(pv)


### Ejercicio 5
Crea estos pivots:
1. `Branch` × `Product line` con suma de `Total`.
2. `City` × `Gender` con media de `Rating`.
3. `Branch` × `Payment` con suma de `gross income`.

In [18]:
# TODO
import pandas as pd
from IPython.display import display

# Cargar el dataset
df = pd.read_csv('supermarket_sales - Sheet1.csv')

# 1. Branch × Product line con suma de Total
pivot_1 = df.pivot_table(
    values='Total', 
    index='Branch', 
    columns='Product line', 
    aggfunc='sum'
)

# 2. City × Gender con media de Rating
pivot_2 = df.pivot_table(
    values='Rating', 
    index='City', 
    columns='Gender', 
    aggfunc='mean'
)

# 3. Branch × Payment con suma de gross income
pivot_3 = df.pivot_table(
    values='gross income', 
    index='Branch', 
    columns='Payment', 
    aggfunc='sum'
)

# Mostrar resultados con formato
print("### 1. Ventas Totales por Sucursal y Línea de Producto")
display(pivot_1)

print("\n### 2. Rating Promedio por Ciudad y Género")
display(pivot_2)

print("\n### 3. Ingresos Brutos por Sucursal y Método de Pago")
display(pivot_3)


### 1. Ventas Totales por Sucursal y Línea de Producto


Product line,Electronic accessories,Fashion accessories,Food and beverages,Health and beauty,Home and lifestyle,Sports and travel
Branch,,,,,,
A,"18,317.11","16,332.51","17,163.10","12,597.75","22,417.20","19,372.70"
B,"17,051.44","16,413.32","15,214.89","19,980.66","17,549.16","19,988.20"
C,"18,968.97","21,560.07","23,766.85","16,615.33","13,895.55","15,761.93"



### 2. Rating Promedio por Ciudad y Género


Gender,Female,Male
City,,
Mandalay,6.88,6.76
Naypyitaw,7.16,6.97
Yangon,6.84,7.20



### 3. Ingresos Brutos por Sucursal y Método de Pago


Payment,Cash,Credit card,Ewallet
Branch,,,
A,"1,608.63","1,575.94","1,872.59"
B,"1,682.83","1,778.33","1,595.87"
C,"2,051.71","1,444.16","1,769.30"


---
## 7. Extensión opcional

Si sobra tiempo, puedes cerrar la sesión con un mini reporte.

### Opcional: `build_report`
Implementa `build_report(df)` para que devuelva un diccionario con estas tablas:
- `sales_by_city_branch`
- `profit_by_product_line`
- `ticket_band_mix`
- `weekday_hour_heatmap`

In [21]:
import pandas as pd
from IPython.display import display

def build_report(data: pd.DataFrame) -> dict:
    # Copiamos el dataframe para no modificar el original fuera de la función
    df = data.copy()
    
    # Preprocesamiento de fechas y horas
    df['Date'] = pd.to_datetime(df['Date'])
    df['Time'] = pd.to_datetime(df['Time']).dt.hour
    df['weekday'] = df['Date'].dt.day_name()
    
    # 1. sales_by_city_branch: Suma de Total por Ciudad y Sucursal
    sales_by_city_branch = df.pivot_table(
        index='City', 
        columns='Branch', 
        values='Total', 
        aggfunc='sum'
    )
    
    # 2. profit_by_product_line: Suma de gross income por línea de producto
    profit_by_product_line = df.groupby('Product line')[['gross income']].sum().sort_values(by='gross income', ascending=False)
    
    # 3. ticket_band_mix: Relación entre el tipo de cliente y el tamaño del ticket
    # Creamos categorías (bandas) basadas en el Total de la compra
    df['ticket_band'] = pd.cut(
        df['Total'], 
        bins=[0, 100, 500, 1000, float('inf')], 
        labels=['Small', 'Medium', 'Large', 'Very Large']
    )
    ticket_band_mix = pd.crosstab(df['ticket_band'], df['Customer type'])
    
    # 4. weekday_hour_heatmap: Matriz de ventas para identificar picos de tráfico
    weekday_hour_heatmap = df.pivot_table(
        index='weekday', 
        columns='Time', 
        values='Total', 
        aggfunc='sum'
    ).fillna(0)
    
    # Ordenar los días de la semana de lunes a domingo
    dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    weekday_hour_heatmap = weekday_hour_heatmap.reindex(dias)
    
    # Retornar el diccionario con todas las tablas
    return {
        "sales_by_city_branch": sales_by_city_branch,
        "profit_by_product_line": profit_by_product_line,
        "ticket_band_mix": ticket_band_mix,
        "weekday_hour_heatmap": weekday_hour_heatmap
    }

# --- Ejemplo de uso ---
df_sales = pd.read_csv('supermarket_sales - Sheet1.csv')
reporte = build_report(df_sales)
display(reporte['sales_by_city_branch'])

C:\Users\Adrian\AppData\Local\Temp\ipykernel_26788\1523256321.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Time'] = pd.to_datetime(df['Time']).dt.hour


Branch,A,B,C
City,,,
Mandalay,NaN,"106,197.67",NaN
Naypyitaw,NaN,NaN,"110,568.71"
Yangon,"106,200.37",NaN,NaN
